In [ ]:
!git clone https://github.com/devangelista2/IPPy.git

In [ ]:
!pip install numpy torch torchvision numba astra-toolbox scikit-image Pillow matplotlib cupy-cuda12x tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import os
import time

import glob
import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

from nn.diffusion import (
    DiffusionUNet,
    EMA,
    cosine_beta_schedule,
    ddim_sample,
    ddpm_sample,
    denormalize_to_01,
    extract,
)

# In Colab, we replace the __file__ logic with the actual path to the cloned repository
repo_path = os.path.abspath('/content/IPPy')
if repo_path not in sys.path:
    sys.path.append(repo_path)

from IPPy import operators, utilities


book_root = Path('.').resolve()
weights_dir = book_root / 'weights'
weights_dir.mkdir(exist_ok=True)
dataset_dir = book_root / "drive" / "MyDrive" / "Mayo"

In [ ]:
class MayoDataset(Dataset):
    def __init__(self, data_path, data_shape=64):
        super().__init__()
        self.fname_list = sorted(glob.glob(f'{data_path}/*/*.png'))
        self.transform = transforms.Compose([
            transforms.Resize((data_shape, data_shape), antialias=True),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])

    def __len__(self):
        return len(self.fname_list)

    def __getitem__(self, idx):
        x = Image.open(self.fname_list[idx]).convert('L')
        return self.transform(x)


device = get_device()
batch_size = 32 if device == 'cuda' else 8
train_dataset = MayoDataset(data_path=str(dataset_dir / 'train'), data_shape=64)
test_dataset = MayoDataset(data_path=str(dataset_dir / 'test'), data_shape=64)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device == 'cuda'),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device == 'cuda'),
)


def show_batch(batch, title, ncols=6):
    images = denormalize_to_01(batch[:ncols]).cpu()
    fig, axes = plt.subplots(1, len(images), figsize=(2.2 * len(images), 2.2))
    axes = axes if len(images) > 1 else [axes]
    for ax, image in zip(axes, images):
        ax.imshow(image.squeeze(), cmap='gray')
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


print('Device:', device)
print('Training images:', len(train_dataset))
print('Test images:', len(test_dataset))
print('Weights directory:', weights_dir)

sample_batch = next(iter(train_loader))
show_batch(sample_batch, 'Training slices (normalized to [-1, 1] internally)')

In [ ]:
def make_beta_schedule(num_steps):
    return cosine_beta_schedule(num_steps)

num_diffusion_steps = 400
betas = make_beta_schedule(num_diffusion_steps)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

x0 = test_dataset[0].unsqueeze(0)
steps_to_show = [0, 40, 120, 240, 399]

fig, axes = plt.subplots(1, len(steps_to_show), figsize=(15, 3))
for ax, step in zip(axes, steps_to_show):
    t = torch.tensor([step], dtype=torch.long)
    noise = torch.randn_like(x0)
    x_t = extract(alpha_bars.sqrt(), t, x0.shape) * x0 + extract((1 - alpha_bars).sqrt(), t, x0.shape) * noise
    ax.imshow(denormalize_to_01(x_t).squeeze(), cmap='gray')
    ax.set_title(f't = {step}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
model = DiffusionUNet(
    in_ch=1,
    base_ch=64,
    channel_mults=(1, 2, 4),
    time_dim=256,
    dropout=0.05,
    attn_levels=(1, 2),
)
num_params = sum(param.numel() for param in model.parameters())
print(model)
print(f'Trainable parameters: {num_params / 1e6:.2f}M')

In [ ]:
torch.manual_seed(0)

target_epochs = 60
learning_rate = 2e-4
weight_decay = 1e-4
ema_decay = 0.9995
grad_clip = 1.0
force_restart = False

model = DiffusionUNet(
    in_ch=1,
    base_ch=64,
    channel_mults=(1, 2, 4),
    time_dim=256,
    dropout=0.05,
    attn_levels=(1, 2),
).to(device)
ema = EMA(model, decay=ema_decay)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=target_epochs)
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
autocast_device = 'cuda' if device == 'cuda' else 'cpu'

history = []
start_epoch = 0
best_loss = float('inf')
weights_path = weights_dir / 'DDPMDenoiser.pth'
raw_weights_path = weights_dir / 'DDPMDenoiser_raw.pth'
checkpoint_path = weights_dir / 'DDPMDenoiser.ckpt'

if checkpoint_path.exists() and not force_restart:
    try:
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        model.load_state_dict(checkpoint['model'])
        ema.load_state_dict(checkpoint['ema'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        scheduler.load_state_dict(checkpoint['scheduler'])
        scaler.load_state_dict(checkpoint['scaler'])
        for state in optimizer.state.values():
            for key, value in state.items():
                if isinstance(value, torch.Tensor):
                    state[key] = value.to(device)
        history = checkpoint.get('history', [])
        start_epoch = checkpoint.get('epoch', -1) + 1
        best_loss = checkpoint.get('best_loss', float('inf'))
        print(f'Resuming diffusion training from epoch {start_epoch + 1}.')
    except Exception as exc:
        print(f'Ignoring incompatible checkpoint: {exc}')
        start_epoch = 0
        history = []
        best_loss = float('inf')
elif weights_path.exists() and not force_restart:
    try:
        ema_state = torch.load(weights_path, map_location='cpu')
        model.load_state_dict(ema_state)
        ema.shadow.load_state_dict(ema_state)
        start_epoch = target_epochs
        history = []
        print(f'Found existing EMA weights at {weights_path}. Skipping training.')
    except Exception as exc:
        print(f'Ignoring incompatible EMA weights: {exc}')
        start_epoch = 0
        history = []
        best_loss = float('inf')
else:
    print('Starting diffusion training from scratch.')

for epoch in range(start_epoch, target_epochs):
    model.train()
    epoch_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f'DDPM epoch {epoch + 1}/{target_epochs}', leave=True)

    for step, x0_batch in enumerate(progress_bar, start=1):
        x0_batch = x0_batch.to(device, non_blocking=(device == 'cuda'))
        t = torch.randint(0, num_diffusion_steps, (x0_batch.shape[0],), device=device)
        noise = torch.randn_like(x0_batch)
        x_t = extract(alpha_bars.sqrt(), t, x0_batch.shape) * x0_batch + extract((1 - alpha_bars).sqrt(), t, x0_batch.shape) * noise

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=autocast_device, dtype=torch.float16, enabled=(device == 'cuda')):
            noise_pred = model(x_t, t)
            loss = F.mse_loss(noise_pred, noise)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)

        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=f'{loss.item():.5f}', avg=f'{epoch_loss / step:.5f}')

    epoch_loss /= len(train_loader)
    history.append(epoch_loss)
    scheduler.step()

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), raw_weights_path)
        torch.save(ema.shadow.state_dict(), weights_path)

    checkpoint = {
        'epoch': epoch,
        'model': model.state_dict(),
        'ema': ema.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'history': history,
        'best_loss': best_loss,
        'config': {
            'target_epochs': target_epochs,
            'learning_rate': learning_rate,
            'weight_decay': weight_decay,
            'ema_decay': ema_decay,
            'grad_clip': grad_clip,
            'num_diffusion_steps': num_diffusion_steps,
        },
    }
    torch.save(checkpoint, checkpoint_path)

if not weights_path.exists():
    torch.save(ema.shadow.state_dict(), weights_path)

sample_model = DiffusionUNet(
    in_ch=1,
    base_ch=64,
    channel_mults=(1, 2, 4),
    time_dim=256,
    dropout=0.05,
    attn_levels=(1, 2),
).to(device)
sample_model.load_state_dict(torch.load(weights_path, map_location='cpu'))
sample_model.eval()

if history:
    plt.figure(figsize=(5, 3))
    plt.plot(history)
    plt.title('DDPM training loss')
    plt.xlabel('Epoch')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
ddpm_samples = ddpm_sample(
    sample_model,
    num_samples=8,
    image_shape=(1, 64, 64),
    betas=betas,
    alphas=alphas,
    alpha_bars=alpha_bars,
    device=device,
)

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, image in zip(axes.flat, denormalize_to_01(ddpm_samples).cpu()):
    ax.imshow(image.squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Samples generated with DDPM sampling')
plt.tight_layout()
plt.show()

In [ ]:
ddim_samples = ddim_sample(
    sample_model,
    num_samples=8,
    image_shape=(1, 64, 64),
    alpha_bars=alpha_bars,
    device=device,
    sample_steps=50,
    eta=0.0,
)

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, image in zip(axes.flat, denormalize_to_01(ddim_samples).cpu()):
    ax.imshow(image.squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Samples generated with DDIM sampling (50 steps)')
plt.tight_layout()
plt.show()